In [1]:
from collections import defaultdict
from datetime import datetime
import hashlib
import json
import os

In [2]:
ORIGIN = "/home/tteuz/Desktop/mestrado/meta_pklot/datasets/organized/PKLot"
SIAMESE = "/home/tteuz/Desktop/mestrado/meta_pklot/datasets/siamese/PKLot"

In [3]:
def get_artifacts(json_path):
    with open(json_path, "r") as file:
        data = json.load(file)

    annotations = defaultdict(list)
    for annotation in data["annotations"]:
        # if annotation["car_id"] != -1 and annotation["car_id"] != -2:
        annotations[annotation["image_id"]].append(annotation)

    return sorted(data["images"], key=lambda x: datetime(*x["date"], *x["time"])), dict(annotations)

In [4]:
def get_spot_id(annotation):
    rotated_rect = annotation["best_rotated_rect"]
    flat = [round(x, 4) for item in rotated_rect for x in (item if isinstance(item, list) else [item])]

    rect_string = ','.join(map(str, flat))
    return hashlib.sha256(rect_string.encode()).hexdigest()[:10]

In [5]:
def get_dates(ds_path):
    dates = os.listdir(ds_path)
    formated_dates = [(datetime.strptime(date, '%Y-%m-%d').date(), f"{ds_path}/{date}") for date in dates]
    
    return sorted(formated_dates, key=lambda x: x[0])

In [6]:
jsons = [file for file in os.listdir(ORIGIN) if ".json" in file]
jsons = sorted(jsons)

for file in jsons:
    images, annotations = get_artifacts(f"{ORIGIN}/{file}")
    subset = file.replace(".json", "").upper()

    spots_per_image, unique_spot_ids = [], []
    for _, img_annotations in annotations.items():
        spots_per_image.append(len(img_annotations))
        unique_spot_ids.extend([get_spot_id(ann) for ann in img_annotations])
    unique_spot_ids = set(unique_spot_ids)

    print(f"Subset: {subset}")
    print(f"Max: {max(spots_per_image)} - Min: {min(spots_per_image)} - Unique spots: {len(unique_spot_ids)} - Images: {len(spots_per_image)}\n")

Subset: PUCPR_SPOTS
Max: 195 - Min: 195 - Unique spots: 195 - Images: 4474

Subset: UFPR04_SPOTS
Max: 37 - Min: 35 - Unique spots: 110 - Images: 3791

Subset: UFPR05_SPOTS
Max: 46 - Min: 46 - Unique spots: 46 - Images: 4135



In [7]:
subsets = [os.path.join(SIAMESE, subset) for subset in os.listdir(SIAMESE)]
subsets = sorted(subsets)

for subset in subsets:
    dates = get_dates(subset)
    images = [image for _, date in dates for image in os.listdir(date)]

    spots_per_image = defaultdict(list)
    for image in images:
        image_id = image.split("#")[-3]
        spots_id = image.split("#")[-2]
        spots_per_image[image_id].append(spots_id)

    spots_per_image = [len(set(spots)) for spots in spots_per_image.values()]
    
    print(f"Subset: {subset.split('/')[-1]}")
    print(f"Max: {max(spots_per_image)} - Min: {min(spots_per_image)} - Days: {len(spots_per_image)}\n")


Subset: PUCPR
Max: 190 - Min: 1 - Days: 4159

Subset: UFPR04
Max: 37 - Min: 1 - Days: 2619

Subset: UFPR05
Max: 46 - Min: 1 - Days: 3394

